# Week 3 Exercise – Synthetic Data Generator (iyanuashiri)

Generate synthetic datasets via **OpenRouter** (OpenAI library), with **Gradio** UI and **model switching**.

- **Schema** – define columns (name, type, example)
- **Description** – short business/use-case description
- **Model** – dropdown of OpenRouter models
- **Rows & temperature** – control size and variety
- **Output** – preview table + download as CSV or JSONL

## Imports and environment

In [1]:
import os
import json
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd

load_dotenv(override=True)
try:
    from decouple import config
    OPEN_ROUTER_API_KEY = config("OPEN_ROUTER_API_KEY")
except Exception:
    OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")

OPEN_ROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=OPEN_ROUTER_BASE_URL, api_key=OPEN_ROUTER_API_KEY)

# OpenRouter models for dropdown (good for structured data generation)
MODELS = [
    "openai/gpt-4o-mini",
    "openai/gpt-4o",
    "anthropic/claude-3.5-haiku",
    "anthropic/claude-3.5-sonnet",
    "meta-llama/llama-3.1-8b-instruct",
    "google/gemini-2.0-flash-001",
]

## Schema and prompts

In [2]:
DEFAULT_SCHEMA = """name (string): full name, e.g. "Jane Doe"
email (string): valid email format
role (string): job title, e.g. "Engineer", "Designer"
department (string): e.g. "Engineering", "Product"
"""

SYSTEM_PROMPT = """You are a synthetic data generator. Given a schema and a short description, you output only valid JSONL: one JSON object per line, with keys matching the schema. No markdown, no code fences, no extra text. Each line must be a single JSON object. Do not repeat the same values across rows; vary the data realistically."""


def build_user_prompt(schema: str, description: str, n_rows: int) -> str:
    return f"""Description: {description or 'General synthetic data'}

Schema (column name and type / example):
{schema.strip()}

Generate exactly {n_rows} rows as JSONL. Output only the JSONL lines, nothing else."""

## Generate and export

In [3]:
def parse_jsonl(text: str) -> list[dict]:
    """Extract JSON objects from model output (strip markdown/code blocks if present)."""
    lines = []
    for line in text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith("```"):
            continue
        try:
            lines.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return lines


def generate_dataset(schema: str, description: str, model_id: str, n_rows: int, temperature: float, file_format: str):
    """Call OpenRouter, parse JSONL, return (status, dataframe, download_path)."""
    if not OPEN_ROUTER_API_KEY:
        return "Set OPEN_ROUTER_API_KEY in .env", None, None
    n_rows = int(n_rows)
    if n_rows < 5 or n_rows > 200:
        return "Number of rows must be between 5 and 200.", None, None
    if not (schema or "").strip():
        return "Schema cannot be empty.", None, None

    user_prompt = build_user_prompt(schema, description, n_rows)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        response = openrouter.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=float(temperature),
        )
        content = response.choices[0].message.content or ""
    except Exception as e:
        return f"API error: {e}", None, None

    records = parse_jsonl(content)
    if not records:
        return "No valid JSONL rows could be parsed. Try a simpler schema or fewer rows.", None, None

    df = pd.DataFrame(records)
    out_dir = Path(tempfile.gettempdir()) / "synthetic_data"
    out_dir.mkdir(exist_ok=True)
    base = out_dir / "dataset"
    if file_format == "CSV":
        path = base.with_suffix(".csv")
        df.to_csv(path, index=False)
    else:
        path = base.with_suffix(".jsonl")
        with open(path, "w") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")

    status = f"Generated {len(records)} rows. Saved to: {path}"
    return status, df, str(path)

## Gradio UI

In [ ]:
with gr.Blocks(title="Synthetic Data Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Synthetic data generator\nDefine a schema, describe the use case, pick a model, and generate CSV or JSONL.")
    with gr.Row():
        schema_in = gr.Textbox(
            label="Schema (one column per line: name (type) example)",
            value=DEFAULT_SCHEMA,
            lines=6,
        )
        description_in = gr.Textbox(
            label="Description / use case",
            placeholder="e.g. Employees at a tech startup",
            lines=2,
        )
    with gr.Row():
        model_in = gr.Dropdown(choices=MODELS, value=MODELS[0], label="Model")
        n_rows_in = gr.Number(value=10, minimum=5, maximum=200, step=1, label="Number of rows")
        temperature_in = gr.Slider(0.1, 0.9, value=0.6, step=0.1, label="Temperature")
        file_format_in = gr.Radio(["CSV", "JSONL"], value="CSV", label="Output format")
    gen_btn = gr.Button("Generate", variant="primary")
    status_out = gr.Textbox(label="Status", interactive=False)
    preview_out = gr.Dataframe(label="Preview", interactive=False)
    download_out = gr.File(label="Download file")

    gen_btn.click(
        fn=generate_dataset,
        inputs=[schema_in, description_in, model_in, n_rows_in, temperature_in, file_format_in],
        outputs=[status_out, preview_out, download_out],
    )

demo.launch(share=True)